In [43]:
import pandas as pd
import re
import numpy as np
import multiprocessing
import string

from typing import List

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

# Constants and configuration
CORES = multiprocessing.cpu_count()
base_stopwords = list(stopwords.words('english')) + [
    "'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'
]
with open ('utils/exclusion_v2.stop', encoding='utf-8') as f:
    additional_stopwords = [x.strip('\n').lower() for x in f.readlines()]

STOPWORDS = base_stopwords #+ additional_stopwords    

def tokenize_lemmatize_remove_stop_words(text: str) -> List[str]:
    return [wnl.lemmatize(token) for token in word_tokenize(text) if 
            token not in STOPWORDS and
            len(token) > 2 and  # Mots de moins de 2 lettres
            not (bool(re.search(r'\d', token))) and # Mots contenant des chiffres
            not (any(char in string.punctuation for char in token)) # Mots contenant des signes de ponctuation
]

data = pd.read_excel(
    '../data/training_datasets/train_dataset_30pc.xlsx')

dic = {'neutral':0, 'incel': 1}
data['label'] = data['category'].map(dic)

vectorizer = TfidfVectorizer(
    stop_words=STOPWORDS,
    token_pattern=None,
    tokenizer=tokenize_lemmatize_remove_stop_words,
    max_features=15000
)

X = vectorizer.fit_transform(data['text_post'].astype('str'))
y = data['label']

feature_names = vectorizer.get_feature_names_out()

model = LogisticRegression(
    n_jobs=CORES,
)

model.fit(X, y)

coefficients = model.coef_[0]
feature_importance = pd.DataFrame({'Trait': feature_names, 'Coefficient': coefficients})
feature_importance_incels = feature_importance.sort_values('Coefficient', ascending=False)[:20]
feature_importance_incels

,Trait,Coefficient
6459,incel,6.598172
1990,chad,6.531938
14502,woman,5.730333
13658,ugly,5.701095
6461,incels,5.530230
8767,normies,4.689418
395,alone,4.670280
14105,virgin,4.569696
7596,loneliness,4.555348
10688,relationship,4.418096


In [44]:
feature_importance_neutrals = feature_importance.sort_values('Coefficient', ascending=True)[:20]
feature_importance_neutrals['Coefficient'] = feature_importance_neutrals['Coefficient'].transform(lambda x : np.abs(x))
feature_importance_neutrals

,Trait,Coefficient
12970,team,3.570226
7150,kink,3.085390
9680,player,2.836032
6173,host,2.740311
2024,character,2.517602
13901,use,2.495985
619,appreciated,2.472400
13401,trade,2.453625
11427,season,2.416224
6878,item,2.401638


In [45]:
feature_importance_incels = feature_importance_incels.to_dict('records')
feature_importance_neutrals = feature_importance_neutrals.to_dict('records')

feature_importance = []
for i in range(20):
    feature_importance.append({
        'Trait_incel':feature_importance_incels[i]['Trait'],
        'Coefficient_incel':feature_importance_incels[i]['Coefficient'],
        'Trait_neutre':feature_importance_neutrals[i]['Trait'],
        'Coefficient_neutre':feature_importance_neutrals[i]['Coefficient']
    })


feature_importance = pd.DataFrame(feature_importance)
feature_importance

,Trait_incel,Coefficient_incel,Trait_neutre,Coefficient_neutre
0,incel,6.598172,team,3.570226
1,chad,6.531938,kink,3.085390
2,woman,5.730333,player,2.836032
3,ugly,5.701095,host,2.740311
4,incels,5.530230,character,2.517602
5,normies,4.689418,use,2.495985
6,alone,4.670280,appreciated,2.472400
7,virgin,4.569696,trade,2.453625
8,loneliness,4.555348,season,2.416224
9,relationship,4.418096,item,2.401638


In [46]:
feature_importance.to_csv('../results/top_features_per_class.csv', index=False)